<a href="https://colab.research.google.com/github/hoin1357/MachineLearning/blob/main/3%EC%B0%A8%EA%B3%BC%EC%A0%9C/3%EC%B0%A8_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 서울대공원 방문객 예측 모델 성능 변화 보고서

우선 목표: 실제 혼잡일을 예측 상위 위험일로 잘 올리는 것  
주요 우선 지표: NDCG@20, Recall@20, Precision@20  

실행 방법: 모델 성능을 확인하면서 하이퍼파라미터를 반복적으로 조정했다. 단순히 오차가 가장 작은 모델만 고르지 않고, 실제로 혼잡한 날을 위험일 상위권에 잘 올리는지를 우선 기준으로 삼았다. 딥러닝에서 사용되는 방식과 비슷하게 성능이 더 이상 개선되지 않거나 비슷한 결과가 반복될 때는 추가 조정을 멈추고 가장 안정적인 값을 선택했다.

## 1. 지표 해석 기준

| 지표 | 의미 | 해석 |
| --- | --- | --- |
| MAE | 실제 방문객 수와 예측 방문객 수의 절대 오차 평균 | 낮을수록 좋음 |
| RMSE | 큰 오차에 더 큰 벌점을 주는 평균 오차 | 낮을수록 좋음 |
| MAPE | 실제값 대비 오차율 평균 | 낮을수록 좋음 |
| R2 | 실제 변동성을 모델이 설명하는 정도 | 높을수록 좋음 |
| Precision@20 | 예측 상위 20일 중 실제 혼잡일 비율 | 높을수록 좋음 |
| Recall@20 | 실제 혼잡일 중 예측 상위 20일에 잡힌 비율 | 높을수록 좋음 |
| NDCG@20 | 혼잡한 날을 예측 순위 상단에 잘 배치했는지 | 높을수록 좋음 |

이 프로젝트에서는 단순히 전체 날짜의 평균 오차를 줄이는 것보다, 실제로 사람이 많이 몰리는 날을 놓치지 않는 것이 더 중요했다. 그래서 튜닝 기준은 MAE나 RMSE만 보지 않고, NDCG@20과 Recall@20을 중심으로 잡았다.

## 2. 전체 성능 변화 요약

| 단계 | 설명 | MAE | RMSE | MAPE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 기준 모델 | 하이퍼파라미터 튜닝 전 기준 | 3266.38 | 4620.77 | 176.39 | 0.7312 | 0.55 | 0.7857 | 0.8825 |
| 코로나 가중치 실험 | 2020~2022 데이터 가중치 0.25 적용 | 3160.85 | 4376.30 | 185.10 | 0.7589 | 0.55 | 0.7857 | 0.8942 |
| 코로나 반사실+가중치 실험 | 코로나가 없었다고 가정한 보정값과 원자료 병행 | 3247.17 | 4376.91 | 191.37 | 0.7588 | 0.60 | 0.8571 | 0.9071 |
| 1단계 혼잡일 분류 튜닝 후 | 혼잡일 분류 모델 100회 탐색 결과 | 3160.34 | 4419.26 | 190.01 | 0.7541 | 0.65 | 0.9286 | 0.9399 |
| 2단계 최종 방문객 예측 튜닝 후 | 최종 방문객 수 회귀 모델 100회 탐색 결과 | 3214.38 | 4509.65 | 194.84 | 0.7440 | 0.65 | 0.9286 | 0.9478 |
| 실제 서비스 사용 모델 | 실제 미래 예측 시점에 확보 가능한 feature만 사용 | 3347.89 | 5414.60 | 179.34 | 0.6309 | 0.55 | 0.7857 | 0.8292 |
| 참고: 당일 대공원역 데이터 직접 투입 | 실제 서비스에는 사용할 수 없는 분석용 모델 | 2497.10 | 4069.31 | 135.44 | 0.7915 | 0.65 | 0.9286 | 0.9418 |

## 3. 코로나 기간 데이터 처리

### 처리한 이유

2020년부터 2022년까지는 코로나19로 인해 방문객 수가 평소와 다른 패턴을 보였다. 이 시기에는 사회적 거리두기, 외출 감소, 시설 운영 제한, 단체 방문 감소 같은 요인이 방문객 수에 직접적인 영향을 줬다.  

문제는 이 변화가 날씨, 요일, 공휴일, 계절성 같은 일반적인 방문 패턴으로 설명되기 어렵다는 점이다. 모델이 이 기간을 그대로 학습하면, 코로나 때문에 줄어든 방문객 수를 일반적인 계절 패턴이나 날짜 특성으로 잘못 이해할 수 있다. 그렇게 되면 코로나 이후의 정상적인 방문 수요를 예측할 때 오히려 성능이 떨어질 가능성이 있다.

그래서 코로나 기간을 완전히 삭제하지는 않고, 두 가지 방식으로 따로 실험했다.


### 방법 1. 코로나 기간 가중치 낮추기

첫 번째 방법은 2020~2022년 데이터의 학습 가중치를 낮추는 방식이다.

```text
일반 기간 데이터 가중치 = 1.0
코로나 기간 데이터 가중치 = 0.25
```

이 방식은 코로나 기간 데이터를 버리지는 않지만, 모델이 해당 기간의 특수한 패턴을 너무 강하게 학습하지 않도록 만든다.  
즉, 코로나 기간도 실제로 존재했던 데이터이므로 참고는 하되, 정상적인 방문 패턴을 대표하는 데이터로는 덜 반영되게 한 것이다.

이 방법을 적용한 결과 RMSE는 4620.77에서 4376.30으로 낮아졌고, NDCG@20도 0.8825에서 0.8942로 조금 개선되었다. 전체적으로는 평균 오차와 혼잡일 순위 성능이 모두 기준 모델보다 나아졌다.

### 방법 2. 코로나가 없었다고 가정한 보정값 함께 사용

두 번째 방법은 코로나 기간의 방문객 수를 그대로만 쓰지 않고, 코로나가 없었다면 나왔을 가능성이 있는 방문객 수를 별도로 추정해서 함께 사용하는 방식이다.

처리 흐름은 다음과 같다.

```text
1. 코로나 이전과 이후의 정상 기간 데이터를 기준으로 방문객 패턴을 학습
2. 날짜, 요일, 공휴일, 계절, 날씨 등의 특성을 이용해 코로나 기간의 예상 방문객 수를 추정
3. 실제 코로나 기간 방문객 수와 보정된 방문객 수를 함께 사용
4. 코로나 기간 원자료에는 낮은 가중치를 적용
```

이 방식은 코로나 기간을 단순히 “이상치”로만 보지 않고, 두 가지 정보를 나눠서 반영하려는 목적이 있었다.

- 실제 방문객 수: 코로나라는 특수 상황에서 실제로 발생한 수요
- 보정 방문객 수: 코로나가 없었다면 나왔을 가능성이 있는 일반적인 수요

이렇게 처리한 이유는 코로나 기간을 전부 제거하면 3년치 데이터가 빠져 학습 데이터가 줄어들고, 반대로 그대로 사용하면 비정상적인 방문 패턴을 너무 많이 학습할 수 있기 때문이다. 그래서 원자료와 보정값을 같이 사용하되, 코로나 원자료에는 낮은 가중치를 주는 방식으로 균형을 맞췄다.

### 코로나 처리 결과

| 실험 | MAE | RMSE | MAPE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 기준 모델 | 3266.38 | 4620.77 | 176.39 | 0.7312 | 0.55 | 0.7857 | 0.8825 |
| 코로나 가중치 0.25 | 3160.85 | 4376.30 | 185.10 | 0.7589 | 0.55 | 0.7857 | 0.8942 |
| 코로나 반사실+가중치 | 3247.17 | 4376.91 | 191.37 | 0.7588 | 0.60 | 0.8571 | 0.9071 |

코로나 가중치만 낮춘 경우에는 MAE와 RMSE가 개선되었다. 반면 코로나 보정값을 함께 사용한 경우에는 평균 오차는 크게 좋아지지 않았지만, Precision@20, Recall@20, NDCG@20이 더 좋아졌다.  

이 프로젝트에서는 혼잡일을 상위 위험일로 잘 올리는 것이 중요하므로, 최종 방향은 코로나 기간을 완전히 제거하기보다 가중치를 낮추고 보정값을 함께 검토하는 방식이 더 적합하다고 판단했다.

## 4. 1단계 혼잡일 분류 모델 튜닝

1단계 모델은 날짜별로 혼잡일일 가능성을 먼저 분류하는 역할을 한다. 이 결과는 이후 2단계 방문객 수 예측 모델에 함께 들어간다.

### 최종 선택 파라미터

```python
CLASSIFIER_PARAMS = {
    "loss_function": "Logloss",
    "depth": 6,
    "learning_rate": 0.06,
    "iterations": 800,
    "l2_leaf_reg": 5.0,
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}
```

### 성능 변화

| 비교 | MAE | RMSE | R2 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: |
| 기준 모델 | 3266.38 | 4620.77 | 0.7312 | 0.8825 |
| RMSE 우선 탐색 최고 | 2839.21 | 4188.74 | 0.7791 | 0.8952 |
| 혼잡도 우선 100회 탐색 최고 | 3160.34 | 4419.26 | 0.7541 | 0.9399 |

RMSE를 기준으로 고른 모델은 평균 오차가 더 작았지만, 혼잡일을 상위 위험일로 올리는 성능은 충분하지 않았다. 최종 목적이 혼잡일 탐지였기 때문에 NDCG@20이 높은 설정을 선택했다.

## 5. 데이터 결측값 보정 모델 튜닝

방문객 데이터에는 일부 긴 결측 구간이 있었다. 결측값 보정 모델은 이 구간의 방문객 수를 학습 데이터에 자연스럽게 채우기 위한 보조 모델이다.

### 최종 선택 파라미터

```python
IMPUTATION_MODEL_PARAMS = {
    "loss_function": "RMSE",
    "depth": 6,
    "learning_rate": 0.05,
    "iterations": 700,
    "l2_leaf_reg": 5.0,
    "min_data_in_leaf": 5,
    "random_strength": 1.0,
    "verbose": False,
    "random_seed": 42,
    "allow_writing_files": False,
}
```

### 성능 변화

| 단계 | MAE | RMSE | R2 | NDCG@20 | 판단 |
| --- | ---: | ---: | ---: | ---: | --- |
| 기존 결측값 보정 설정 | 3225.01 | 4313.09 | 0.7658 | 0.9300 | 최종 유지 |
| 최고 후보 | 3225.01 | 4313.09 | 0.7658 | 0.9300 | 기존과 동률 |

결측값 보정 모델은 파라미터를 바꿔도 전체 성능 개선이 없었다. 따라서 기존 설정을 그대로 유지했다.

## 6. 2단계 최종 방문객 수 예측 모델 튜닝

2단계 모델은 최종 방문객 수를 예측하는 모델이다. 앞 단계에서 나온 혼잡일 가능성 정보도 함께 사용했다.

### 최종 선택 파라미터

```python
REGRESSOR_PARAMS = {
    "loss_function": "RMSE",
    "depth": 10,
    "learning_rate": 0.04,
    "iterations": 1200,
    "l2_leaf_reg": 8.0,
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}
```

### 성능 변화

| 단계 | MAE | RMSE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 1단계 튜닝 후 | 3160.34 | 4419.26 | 0.7541 | 0.65 | 0.9286 | 0.9399 |
| 2단계 최종 튜닝 후 | 3214.38 | 4509.65 | 0.7440 | 0.65 | 0.9286 | 0.9478 |

2단계 튜닝 후 MAE와 RMSE는 약간 나빠졌지만, NDCG@20은 0.9399에서 0.9478로 상승했다. 방문객 수를 평균적으로 조금 더 정확히 맞히는 것보다, 혼잡한 날을 상위 위험일로 올리는 쪽을 우선한 결과다.

## 7. 대공원역 방문 데이터 추가

### 방문객 수와 대공원역 승하차의 상관관계

| 구간 | N | Pearson | Spearman | 평균 방문객 | 평균 승하차 |
| --- | ---: | ---: | ---: | ---: | ---: |
| 전체 2015~2025.07 | 3762 | 0.8229 | 0.8247 | 5360.6 | 9926.0 |
| 코로나 전 2015~2019 | 1723 | 0.8636 | 0.8452 | 6854.1 | 12172.0 |
| 코로나 2020~2022 | 1096 | 0.7452 | 0.7834 | 3785.0 | 6782.3 |
| 코로나 후 2023~2025.07 | 943 | 0.7402 | 0.8483 | 4462.7 | 9476.1 |
| 방문객 상위 15% | 565 | 0.6697 | 0.5577 | 20094.5 | 21645.2 |

대공원역 당일 승하차 데이터는 방문객 수와 매우 강한 관계가 있었다. 특히 전체 기간 기준 Spearman 상관계수가 0.8247로 높게 나왔다. 다만 실제 미래 예측 시점에는 예측 대상 날짜의 대공원역 승하차 인원을 미리 알 수 없으므로, 최종 서비스 모델에는 직접 넣지 않았다.

### 지나간 교통량과 날씨의 연관도 비교

| 데이터 | 대표 변수 | Spearman |
| --- | --- | ---: |
| 당일 교통량 | 당일 대공원역 하차 | 0.8253 |
| 과거 교통량 | 1일 전 대공원역 하차 | 0.6216 |
| 과거 교통량 | 7일 전 대공원역 하차 | 0.6187 |
| 날씨 | 18도에서 떨어진 정도 | -0.5939 |
| 날씨 | 너무 추운 날 여부 | -0.4501 |
| 날씨 | 일강수량 | -0.2259 |

당일 교통량은 방문객 수와 가장 높은 연관도를 보였다. 과거 교통량도 날씨보다 높은 편이었지만, 장기 미래 예측에서는 1일 전이나 7일 전 교통량도 아직 알 수 없는 값이 될 수 있다. 그래서 최종 장기 예측 모델에서는 제외했다.

## 8. 대공원역 데이터를 넣었을 때의 성능

| 모델 | MAE | RMSE | MAPE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 2단계 최종 튜닝 모델 | 3214.38 | 4509.65 | 194.84 | 0.7440 | 0.65 | 0.9286 | 0.9478 |
| 당일 대공원역 데이터 직접 투입 | 2497.10 | 4069.31 | 135.44 | 0.7915 | 0.65 | 0.9286 | 0.9418 |

대공원역 데이터를 직접 넣으면 방문객 수 오차는 크게 줄었다. 하지만 이 결과는 예측 대상 날짜의 실제 교통량을 이미 알고 있다고 가정한 것이다. 실제 서비스에서는 이 값을 미리 알 수 없기 때문에, 참고용 성능으로만 사용했다.

## 9. 실제 미래 예측용 모델로 변경

기존 모델은 과거 방문객 수를 이용한 lag, rolling feature를 사용했다. 검증 구간에서는 실제 과거값을 알 수 있기 때문에 성능이 좋게 나올 수 있지만, 긴 미래를 예측할 때는 중간 날짜의 실제 방문객 수를 알 수 없다. 이 경우 앞에서 예측한 값을 다시 다음 예측의 입력으로 사용하게 되고, 예측 기간이 길어질수록 오차가 누적될 수 있다.

그래서 최종 서비스용 모델은 미래 시점에도 안정적으로 확보할 수 있는 변수만 사용하도록 바꿨다.

### 최종 서비스 모델 feature

```text
날짜
요일
주말 여부
공휴일 여부
연휴 여부
방학/성수기 여부
행사 여부
날씨 API 예보 평균기온
날씨 API 예보 강수량
강수 여부
폭우 여부
너무 추움 여부
너무 더움 여부
```

### 성능 변화

| 모델 | MAE | RMSE | MAPE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 2단계 최종 튜닝 모델 | 3214.38 | 4509.65 | 194.84 | 0.7440 | 0.65 | 0.9286 | 0.9478 |
| 실제 미래 예측용 모델 | 3347.89 | 5414.60 | 179.34 | 0.6309 | 0.55 | 0.7857 | 0.8292 |

성능은 하락했다. 하지만 이 결과가 실제 서비스에서 기대할 수 있는 성능에 더 가깝다. 기존 모델은 검증 성능은 좋았지만, 장기 미래 예측에서는 예측값을 다시 입력으로 쓰는 구조라 운영 안정성이 낮았다.

## 10. 최종 판단

### 성능만 보면

```text
당일 대공원역 데이터 직접 투입 모델
> 2단계 하이퍼파라미터 튜닝 모델
> 실제 미래 예측용 모델
```

### 실제 서비스 가능성까지 고려하면

```text
실제 미래 예측용 모델
> 2단계 하이퍼파라미터 튜닝 모델
> 당일 대공원역 데이터 직접 투입 모델
```

대공원역 데이터는 방문객 수와 강한 연관이 있으므로 분석용으로는 매우 의미가 있었다. 다만 장기 미래 예측 서비스에서는 예측 대상 날짜의 실제 교통량을 미리 알 수 없기 때문에 직접 입력 변수로 사용하기 어렵다. 현재 조건에서는 날짜, 공휴일, 행사, 날씨 예보처럼 미래에도 확보 가능한 정보를 중심으로 모델을 운영하는 것이 가장 안전하다.기존 모델은 미래를 예측할때 기존 자신의 예측 결과를 데이터로 학습해서 사용했다. 하지만 이번 모델을 개선하면서 그 문제를 해결했다.

## 11. 생성/참고 산출물

| 산출물 | 내용 |
| --- | --- |
| `backend/data/artifacts/classifier_random100_summary.md` | 1단계 혼잡일 분류 모델 100회 탐색 기록 |
| `backend/data/artifacts/imputation_tuning_summary.md` | 결측값 보정 모델 튜닝 기록 |
| `backend/data/artifacts/regressor_random100_summary.md` | 2단계 방문객 수 예측 모델 100회 탐색 기록 |
| `backend/data/artifacts/covid_preprocessing_model_comparison.csv` | 코로나 전처리 실험 비교 |
| `backend/data/artifacts/external_transport/daegongwon_subway_correlation_summary.csv` | 대공원역 승하차와 방문객 상관관계 |
| `backend/data/artifacts/external_transport/traffic_vs_weather_correlation.csv` | 교통 데이터와 날씨 데이터 연관도 비교 |
| `backend/data/artifacts/portable_predictions.csv` | 최종 실제 미래 예측용 캐시 |
